In [49]:
import pandas as pd

In [75]:
solar_cluster_ts=pd.read_csv('solar_clusters_ts.csv')
wind_cluster_ts=pd.read_csv('wind_clusters_ts.csv')

In [93]:
def create_datetime_col(df):
    # Convert 'Unnamed: 0' column to datetime
    df['datetime'] = pd.to_datetime(df['Unnamed: 0'])

    # Drop the older column
    df.drop(columns=['Unnamed: 0'], inplace=True)
    
    # Set datetime column as index
    df.set_index('datetime', inplace=True)
    
    return df

In [99]:
def create_timeslice_from_cluster(cluster_ts_csv,site_name):
    df=pd.read_csv(cluster_ts_csv)
    df=create_datetime_col(df)

    # Improved season configuration with consistent naming convention
    season_config = {
        "Winter1": {"start": "2021-01-01", "end": "2021-03-19", "daytime": {"day": 8, "night": 17}},
        "Spring": {"start": "2021-03-20", "end": "2021-06-19", "daytime": {"day": 6, "night": 20}},
        "Summer": {"start": "2021-06-20", "end": "2021-09-21", "daytime": {"day": 6, "night": 21}},
        "Fall": {"start": "2021-09-22", "end": "2021-12-20", "daytime": {"day": 8, "night": 18}},
        "Winter2": {"start": "2021-12-21", "end": "2021-12-31", "daytime": {"day": 8, "night": 17}}
    }

    # Convert season_config dates to datetime with error handling
    for season, dates in season_config.items():
        try:
            dates["start"] = pd.to_datetime(dates["start"])
            dates["end"] = pd.to_datetime(dates["end"])
        except ValueError as e:
            print(f"Error converting dates for season '{season}': {e}")

    # Improved function for clarity and error handling
    def get_season_time_of_day(datetime):
        for season, dates in season_config.items():
            if dates["start"] <= datetime <= dates["end"]:
                # Handle specific time ranges for day and night
                if 0 <= datetime.hour <= dates["daytime"]["day"]:
                    return f"{season}_N"  # Night
                elif dates["daytime"]["day"] <= datetime.hour <= dates["daytime"]["night"]:
                    return f"{season}_D"  # Day
                else:  # Outside daytime range
                    return f"{season}_N"  # Assuming night for values outside day/night times
        # Handle date outside any season (optional: return specific value or raise an error)
        return "Out_of_Season"  # Example handling

    # Create the 'season_daytime' column with error handling
    try:
        df['season_daytime'] = df.index.map(get_season_time_of_day)
    except TypeError as e:
        print("Error applying function: ", e)

    # Display the DataFrame
    # Group by season and day/night
    grouped_df = df.groupby(['season_daytime']).agg({
        site_name: ['mean', 'std']  # You can add more aggregation functions here
    })

    # Display the grouped DataFrame
    grouped_df['CF_mean'] = grouped_df[site_name]['mean']
    grouped_df
    return grouped_df

In [103]:
solar_cluster_ts_timeslice= create_timeslice_from_cluster('solar_clusters_ts.csv','East Kootenay_1')
winter_cluster_ts_timeslice= create_timeslice_from_cluster('wind_clusters_ts.csv','East Kootenay_1')

In [105]:
winter_cluster_ts_timeslice

East Kootenay_1             CF_mean
                          mean       std          
season_daytime                                    
Fall_D                0.169159  0.117985  0.169159
Fall_N                0.175640  0.111405  0.175640
Out_of_Season         0.125135  0.104402  0.125135
Spring_D              0.095810  0.094947  0.095810
Spring_N              0.121575  0.091158  0.121575
Summer_D              0.068764  0.066839  0.068764
Summer_N              0.089093  0.064632  0.089093
Winter1_D             0.170018  0.110511  0.170018
Winter1_N             0.174122  0.120005  0.174122
Winter2_D             0.156750  0.087454  0.156750
Winter2_N             0.143902  0.097067  0.143902